## 03 - Silver Layer Transformations

Reads from the enforced bronze table (`students_data.team4-chicago.food_inspections_bronze`) and produces a single silver table at **inspection grain** — one row per inspection.

### Output
* **`food_inspections_silver`** — Clean, structured data at the natural source grain. The raw `Violations` text is parsed into a **struct array** column `violations_parsed` containing `{code, description, comment}` per violation. The gold layer will explode this into violation grain for the star schema.

### Transformations Applied
1. **Deduplicate** on `Inspection_ID`
2. **Trim whitespace** on `DBA_Name` and `AKA_Name`
3. **Validate & fix** `License` (zero → null), `Latitude`/`Longitude` (Chicago bounding box)
4. **Standardise** `Facility_Type` (group 520+ values into consistent categories)
5. **Drop** redundant `Location` column
6. **Parse** `Violations` into struct array `violations_parsed` with `code`, `description`, `comment`
7. **Add** `violation_count` for convenience
8. Inspections with no violations → null `violations_parsed` array, `violation_count` = 0

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

BRONZE_TABLE = "students_data.`team4-chicago`.food_inspections_bronze"
SILVER_TABLE = "students_data.`team4-chicago`.food_inspections_silver"

df = spark.table(BRONZE_TABLE)
print(f"Bronze table loaded: {df.count():,} rows, {len(df.columns)} columns")
df.printSchema()

In [0]:
# --- Deduplicate on Inspection_ID ---
total_rows = df.count()
df = df.dropDuplicates(["Inspection_ID"])
print(f"Rows before dedup: {total_rows:,} | After: {df.count():,}")

# --- Trim whitespace on name fields ---
df = (df
    .withColumn("DBA_Name", F.trim(F.col("DBA_Name")))
    .withColumn("AKA_Name", F.trim(F.col("AKA_Name")))
)

# --- Fix License = 0 (invalid) → null ---
zero_license_count = df.filter(F.col("License") == 0).count()
df = df.withColumn(
    "License",
    F.when(F.col("License") == 0, F.lit(None).cast(IntegerType()))
    .otherwise(F.col("License"))
)
print(f"License = 0 set to null: {zero_license_count:,} rows")

# --- Validate Latitude/Longitude within Chicago bounding box ---
# Chicago approx: lat 41.6–42.1, lon -87.94 to -87.5
out_of_bounds = df.filter(
    F.col("Latitude").isNotNull() & (
        (F.col("Latitude") < 41.6) | (F.col("Latitude") > 42.1) |
        (F.col("Longitude") < -87.94) | (F.col("Longitude") > -87.5)
    )
).count()

df = df.withColumn(
    "Latitude",
    F.when(
        (F.col("Latitude") >= 41.6) & (F.col("Latitude") <= 42.1),
        F.col("Latitude")
    )
).withColumn(
    "Longitude",
    F.when(
        (F.col("Longitude") >= -87.94) & (F.col("Longitude") <= -87.5),
        F.col("Longitude")
    )
)
print(f"Lat/Long out of Chicago bounds nulled: {out_of_bounds:,} rows")

# --- Standardise Facility_Type ---
df = df.withColumn("Facility_Type", F.upper(F.trim(F.col("Facility_Type"))))
df = df.withColumn(
    "Facility_Type",
    F.when(F.col("Facility_Type").contains("RESTAURANT"), F.lit("RESTAURANT"))
    .when(F.col("Facility_Type").contains("GROCERY"), F.lit("GROCERY STORE"))
    .when(F.col("Facility_Type").contains("SCHOOL"), F.lit("SCHOOL"))
    .when(F.col("Facility_Type").contains("DAYCARE"), F.lit("DAYCARE"))
    .when(F.col("Facility_Type").contains("CHILDREN"), F.lit("DAYCARE"))
    .when(F.col("Facility_Type").contains("BAKERY"), F.lit("BAKERY"))
    .when(F.col("Facility_Type").contains("LIQUOR"), F.lit("LIQUOR"))
    .when(F.col("Facility_Type").contains("BAR"), F.lit("BAR"))
    .when(F.col("Facility_Type").contains("TAVERN"), F.lit("BAR"))
    .when(F.col("Facility_Type").contains("COFFEE"), F.lit("COFFEE SHOP"))
    .when(F.col("Facility_Type").contains("CATERING"), F.lit("CATERING"))
    .when(F.col("Facility_Type").contains("MOBILE"), F.lit("MOBILE FOOD"))
    .when(F.col("Facility_Type").contains("HOSPITAL"), F.lit("HOSPITAL"))
    .when(F.col("Facility_Type").contains("LONG TERM"), F.lit("LONG TERM CARE"))
    .when(F.col("Facility_Type").contains("ASSISTED"), F.lit("ASSISTED LIVING"))
    .when(F.col("Facility_Type") == "UNKNOWN", F.lit("UNKNOWN"))
    .otherwise(F.col("Facility_Type"))
)
print(f"Distinct Facility Types: {df.select('Facility_Type').distinct().count()}")

# --- Drop redundant Location column ---
df = df.drop("Location")
print(f"Columns after cleanup: {len(df.columns)}")

### Parse Violations into Struct Array

Each row in the bronze table has a pipe-separated `Violations` string with the pattern:
```
{code}. {DESCRIPTION} - Comments: {COMMENT} | {code}. {DESCRIPTION} - Comments: {COMMENT} | ...
```

We split on ` | `, then use `transform()` + regex to parse each element into a struct with:
* **code** (int) — the number before the `.`
* **description** (string) — the standard rule text
* **comment** (string) — the inspector’s free-text observation

The result is stored as a **struct array column** `violations_parsed`, keeping the table at inspection grain. The gold layer will `explode` this into violation-level rows for the star schema.

In [0]:
# --- Split violations string on " | " into an array ---
df = df.withColumn(
    "violations_array",
    F.when(
        F.col("Violations").isNotNull() & (F.trim(F.col("Violations")) != ""),
        F.split(F.col("Violations"), r"\s*\|\s*")
    )
)

# --- Parse each element into a struct {code, description, comment} ---
df = df.withColumn(
    "violations_parsed",
    F.transform(
        F.col("violations_array"),
        lambda v: F.struct(
            F.regexp_extract(v, r"^(\d+)\.", 1).cast(IntegerType()).alias("code"),
            F.trim(F.regexp_extract(v, r"^\d+\.\s*(.+?)\s*-\s*Comments:", 1)).alias("description"),
            F.trim(F.regexp_extract(v, r"-\s*Comments:\s*(.*)", 1)).alias("comment")
        )
    )
)

# --- Add violation_count for convenience ---
df = df.withColumn(
    "violation_count",
    F.when(F.col("violations_parsed").isNotNull(), F.size(F.col("violations_parsed")))
    .otherwise(F.lit(0))
)

# --- Drop raw text and intermediate columns ---
df = df.drop("Violations", "violations_array")

print(f"Silver table row count (inspection grain): {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()
display(df.limit(20))

In [0]:
# Write silver table (inspection grain, violations as struct array)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)

print(f"Silver table written: {SILVER_TABLE}")
print(f"Rows: {spark.table(SILVER_TABLE).count():,}")